# Day 39：Agent 核心概念与纯 Python ReAct 范式实战（无框架）

> **🎙️ 讲师旁白**：欢迎来到第四阶段（Stage 4：Agent 智能体开发）的第一天！今天我们不依赖任何第三方 Agent 框架，用纯 Python 从零构建 **ReAct（Reasoning + Acting）** 引擎（`v0.4.0-day39`），完整掌握 `Thought -> Action -> Action Input -> Observation -> Final Answer` 循环、AST 安全计算器、离线企业知识检索与 `max_steps` 防死循环守卫。

## 1. 环境初始化与导入 `project_iterations/day_39` 模块

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day39_dir = repo_root / "project_iterations" / "day_39"
if str(day39_dir) not in sys.path:
    sys.path.insert(0, str(day39_dir))

from zhishu_agent import (
    ReActAgent,
    ReActStep,
    ToolSpec,
    calculator_tool,
    knowledge_search_tool,
    parse_react_output,
)

print("已加载模块路径:", day39_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_39


## 2. 内置确定性安全工具演示：AST 计算器 & 离线知识检索
验证 `calculator_tool` 禁用 `eval()` 后的安全拦截能力，以及 `knowledge_search_tool` 对企业制度条款的离线召回。

In [2]:
print("正常计算 (128 + 72) * 3 / 4 =", calculator_tool("(128 + 72) * 3 / 4"))
print("除零保护 10 / 0 ->", calculator_tool("10 / 0"))
print("注入拦截 __import__('os') ->", calculator_tool("__import__('os').system('ls')"))
print("\n知识库检索 '高级工程师 差旅 住宿':")
print(knowledge_search_tool("高级工程师 差旅 住宿"))

正常计算 (128 + 72) * 3 / 4 = 150
除零保护 10 / 0 -> 计算错误: 除数不能为零
注入拦截 __import__('os') -> 计算错误: 禁止的语法结构: Call

知识库检索 '高级工程师 差旅 住宿':
[星辰数科差旅住宿报销标准] 星辰数科员工一线城市出差住宿报销上限：普通工程师 350 元/晚，高级工程师 450 元/晚，技术专家及总监 650 元/晚；餐补统一为 100 元/天。


## 3. ReAct 正则协议解析器测试
观察 `parse_react_output` 如何将模型输出拆解为结构化的 `ParsedAction` 或 `ParsedFinalAnswer`。

In [3]:
raw_action = (
    "Thought: 需要先查询差旅住宿标准。\n"
    "Action: knowledge_search\n"
    "Action Input: 高级工程师 差旅 住宿"
)
raw_final = (
    "Thought: 已获取完整信息。\n"
    "Final Answer: 高级工程师出差 4 晚总住宿预算上限为 1800 元。"
)

print("解析 Action ->", parse_react_output(raw_action))
print("解析 Final Answer ->", parse_react_output(raw_final))

解析 Action -> ParsedAction(thought='需要先查询差旅住宿标准。', action='knowledge_search', action_input='高级工程师 差旅 住宿')
解析 Final Answer -> ParsedFinalAnswer(thought='已获取完整信息。', final_answer='高级工程师出差 4 晚总住宿预算上限为 1800 元。')


## 4. 纯 Python ReAct 多跳推理实战
运行完整循环：第 1 步检索制度 -> 第 2 步调用 AST 计算器 -> 第 3 步汇总输出 `Final Answer`。

In [4]:
agent = ReActAgent(max_steps=5)
trace = agent.run("请查询高级工程师的差旅住宿标准，并计算出差 4 晚的总住宿预算。")
print(trace.format_trace())

Query: 请查询高级工程师的差旅住宿标准，并计算出差 4 晚的总住宿预算。
------------------------------------------------------------
[Step 1]
  Thought: 需要先查询星辰数科的差旅住宿报销标准，再根据天数计算总预算。
  Action: knowledge_search
  Action Input: 高级工程师 差旅 住宿 报销 标准
  Observation: [星辰数科差旅住宿报销标准] 星辰数科员工一线城市出差住宿报销上限：普通工程师 350 元/晚，高级工程师 450 元/晚，技术专家及总监 650 元/晚；餐补统一为 100 元/天。
[Step 2]
  Thought: 已查到高级工程师住宿标准为 450 元/晚，接下来使用计算器计算 4 晚的总预算。
  Action: calculator
  Action Input: 450 * 4
  Observation: 1800
[Step 3]
  Thought: 已经拿到精确的计算结果，可以汇总输出最终答案。
  Final Answer: 根据星辰数科差旅制度，高级工程师住宿报销上限为 450 元/晚，总住宿预算上限为 1800 元。
------------------------------------------------------------
Stopped Reason: final_answer
Final Answer: 根据星辰数科差旅制度，高级工程师住宿报销上限为 450 元/晚，总住宿预算上限为 1800 元。


## 5. `max_steps` 防死循环守卫与自定义工具扩展
演示当规划器陷入死循环时，`max_steps` 如何安全熔断；同时展示动态注册 `get_date` 工具。

In [5]:
def infinite_planner(query: str, steps: list[ReActStep]) -> str:
    return "Thought: 再算一次\nAction: calculator\nAction Input: 1 + 1"

guarded_agent = ReActAgent(planner=infinite_planner, max_steps=3)
guard_trace = guarded_agent.run("死循环测试")
print("熔断状态:", guard_trace.stopped_reason, "| 步数:", len(guard_trace.steps))
print("熔断提示:", guard_trace.final_answer)

# 注册自定义 get_date 工具
agent.register_tool(ToolSpec(name="get_date", description="返回当前日期", func=lambda _: "2026-11-13"))
print("已注册工具列表:", sorted(agent.tools.keys()))

熔断状态: max_steps | 步数: 3
熔断提示: 已达到最大推理步数限制 (max_steps=3)，任务安全中止。
已注册工具列表: ['calculator', 'get_date', 'knowledge_search']
